# CROPSPECTRA — Phase A: Dataset Inspection Notebook

**Project:** CROPSPECTRA — Autonomous Crop Inspection Rover  
**Phase:** A — Dataset Investigation  
**Reference Paper:** Rice-Fusion (IEEE Access, DOI: 10.1109/ACCESS.2022.3140815)  

---

## How to Use This Notebook

1. Place your dataset CSV file in the **same folder** as this notebook  
2. Update the `DATASET_FILE` variable in Cell 1 to match your filename  
3. Run each cell in order (Shift+Enter)  
4. Read the output and markdown comments carefully — they explain what you're seeing  

If you are using **Google Colab**, upload the CSV first using the file browser on the left side panel.

---

> **IMPORTANT:** This notebook does not train any model. Its sole purpose is to understand and document the dataset. We never skip this step.


## Cell 0: Install / Import Libraries

These are all standard Python data-science libraries. Most are pre-installed in Google Colab and Anaconda environments.


In [ ]:
# CELL 0 — Install and import libraries
# Uncomment the line below if running in Google Colab and a library is missing
# !pip install pandas numpy matplotlib seaborn scipy

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import os

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 100

print('All libraries imported successfully.')
print(f'pandas version: {pd.__version__}')
print(f'numpy version:  {np.__version__}')

## Cell 1: Configuration

**Update `DATASET_FILE`** below with the exact name of your CSV file.  
Also update `LABEL_COLUMN` if your dataset uses a different column name for the disease labels.


In [ ]:
# CELL 1 — Configuration

# UPDATE THIS with your actual CSV file name
DATASET_FILE = 'rice_disease_data.csv'

# UPDATE THIS if your label column has a different name
# Common alternatives: 'label', 'disease', 'class', 'target', 'Category'
LABEL_COLUMN = 'label'

# The six sensor features from the Rice-Fusion paper.
# Update column names if your CSV uses different names.
EXPECTED_FEATURES = [
    'temperature',
    'humidity',
    'soil_moisture',
    'nitrogen',
    'phosphorus',
    'potassium'
]

# The five expected class names.
EXPECTED_CLASSES = [
    'Healthy',
    'Rice Blast',
    'Bacterial Leaf Blight',
    'Brown Spot',
    'Sheath Blight'
]

print(f'Dataset file: {DATASET_FILE}')
print(f'Label column: {LABEL_COLUMN}')
print(f'Expected features: {EXPECTED_FEATURES}')

## Cell 2: Load the Dataset

We load the CSV file and immediately check shape, column names, and the first few rows.

**What to look for:** Does the file load? Are column names correct? Does data look numeric?


In [ ]:
# CELL 2 — Load the dataset

if not os.path.exists(DATASET_FILE):
    raise FileNotFoundError(
        f"\nFile '{DATASET_FILE}' not found in: {os.getcwd()}\n"
        f"Files present: {os.listdir('.')}\n\n"
        "Please upload the CSV and update DATASET_FILE in Cell 1."
    )

df = pd.read_csv(DATASET_FILE)

print('=' * 60)
print('DATASET LOADED SUCCESSFULLY')
print('=' * 60)
print(f'Shape: {df.shape[0]} rows x {df.shape[1]} columns')
print()
print('Column names and data types:')
for i, col in enumerate(df.columns):
    print(f'  [{i}] {col}  ({df[col].dtype})')
print()
print('First 5 rows:')
df.head()

## Cell 3: Verify Feature and Label Columns

We check whether the expected columns from the Rice-Fusion paper are present.


In [ ]:
# CELL 3 — Verify feature and label columns

all_columns_lower = df.columns.str.lower().tolist()

print('=== Feature Column Check ===')
found_features = []
missing_features = []
for feat in EXPECTED_FEATURES:
    if feat.lower() in all_columns_lower:
        actual_col = df.columns[all_columns_lower.index(feat.lower())]
        found_features.append(actual_col)
        print(f'  [OK] {feat:25s} found as "{actual_col}"')
    else:
        missing_features.append(feat)
        print(f'  [MISSING] {feat}')

print()
print('=== Label Column Check ===')
if LABEL_COLUMN.lower() in all_columns_lower:
    actual_label_col = df.columns[all_columns_lower.index(LABEL_COLUMN.lower())]
    print(f'  [OK] "{LABEL_COLUMN}" found as "{actual_label_col}"')
else:
    actual_label_col = None
    print(f'  [MISSING] Label column "{LABEL_COLUMN}" not found')
    print(f'  Available columns: {list(df.columns)}')

if missing_features:
    print(f'\nWARNING: {len(missing_features)} expected feature(s) not found: {missing_features}')
    print('Update EXPECTED_FEATURES in Cell 1 to match your actual column names')
else:
    print('\nAll expected features found.')

## Cell 4: Target Class Distribution

We count samples per class. For the Rice-Fusion dataset, we expect ~640 samples per class (3200 total, balanced).

**What to look for:** Are all 5 classes present? Is the distribution roughly balanced?


In [ ]:
# CELL 4 — Class distribution

if actual_label_col is None:
    print('Cannot check class distribution — label column not found. Fix Cell 3 first.')
else:
    class_counts = df[actual_label_col].value_counts()
    class_pct    = (class_counts / len(df) * 100).round(2)
    
    print('=== Class Distribution ===')
    print(f'Total samples: {len(df)}')
    print(f'Unique classes: {df[actual_label_col].nunique()}')
    print()
    print(f'{"Class":<30} {"Count":>8} {"Percent":>10}')
    print('-' * 52)
    for cls, cnt in class_counts.items():
        print(f'{str(cls):<30} {cnt:>8} {class_pct[cls]:>9.2f}%')
    
    print()
    found_classes    = set(class_counts.index.astype(str))
    expected_classes = set(EXPECTED_CLASSES)
    missing_cls   = expected_classes - found_classes
    if missing_cls:
        print(f'WARNING: Expected classes NOT found: {missing_cls}')
        print('(Class names might be spelled differently in your CSV)')
    else:
        print('All five expected classes found.')
    
    imbalance_ratio = class_counts.max() / class_counts.min()
    print(f'Imbalance ratio (max/min): {imbalance_ratio:.2f}x')
    if imbalance_ratio > 3:
        print('WARNING: Significant imbalance. Address in preprocessing (Phase C).')
    
    # Bar chart
    fig, ax = plt.subplots(figsize=(8, 4))
    bars = ax.bar(class_counts.index.astype(str), class_counts.values,
                  color=sns.color_palette('muted', len(class_counts)))
    ax.bar_label(bars, fmt='%d')
    ax.set_title('Class Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel('Disease Class')
    ax.set_ylabel('Sample Count')
    plt.xticks(rotation=30, ha='right')
    plt.tight_layout()
    plt.savefig('class_distribution.png', dpi=120)
    plt.show()

## Cell 5: Data Types and Descriptive Statistics

Checks data types and computes min, max, mean, std for each feature.  
Flags values outside physically plausible sensor ranges.


In [ ]:
# CELL 5 — Descriptive statistics and range checks

print('=== Data Types ===')
print(df.dtypes)
print()
print('=== Descriptive Statistics ===')
print(df.describe().round(3).to_string())
print()

# Plausible sensor ranges for rice crops
reference_ranges = {
    'temperature':   (15, 40,  'deg C'),
    'humidity':      (50, 100, '%'),
    'soil_moisture': (10, 100, '%'),
    'nitrogen':      (0,  200, 'mg/kg'),
    'phosphorus':    (0,  100, 'mg/kg'),
    'potassium':     (0,  500, 'mg/kg'),
}

print('=== Plausible Range Checks ===')
print(f'{"Feature":<18} {"Expected Min":>12} {"Expected Max":>12} {"Actual Min":>10} {"Actual Max":>10}')
print('-' * 68)
for feat, (lo, hi, unit) in reference_ranges.items():
    col_lower_list = [c.lower() for c in df.columns]
    if feat.lower() in col_lower_list:
        actual_col = df.columns[col_lower_list.index(feat.lower())]
        a_min = df[actual_col].min()
        a_max = df[actual_col].max()
        flag = '  <- CHECK RANGE' if (a_min < lo or a_max > hi) else ''
        print(f'{feat:<18} {lo:>12} {hi:>12} {a_min:>10.2f} {a_max:>10.2f}{flag}')
    else:
        print(f'{feat:<18} {lo:>12} {hi:>12}   (column not found)')

## Cell 6: Missing Values and Duplicate Rows


In [ ]:
# CELL 6 — Missing values and duplicates

print('=== Missing Values ===')
missing = df.isnull().sum()
missing_df = missing[missing > 0].to_frame('Count')
missing_df['Percent'] = (missing_df['Count'] / len(df) * 100).round(2)

if missing_df.empty:
    print('No missing values found.')
else:
    print(missing_df)
    print('Will be handled in Phase C (preprocessing).')

print()
print('=== Duplicate Rows ===')
n_dup = df.duplicated().sum()
print(f'Duplicate rows: {n_dup} ({n_dup/len(df)*100:.2f}%)')
if n_dup > 0:
    print('Sample duplicates:')
    print(df[df.duplicated(keep=False)].head(6).to_string())
    print('NOTE: Duplicates can cause data leakage if the same row appears in both train and test sets.')

## Cell 7: Feature Distributions (KDE plots by class)

Plots the distribution of each feature, coloured by disease class.  
Good class separation in any feature = that feature is informative for the model.


In [ ]:
# CELL 7 — Feature distributions by class

numeric_cols = [c for c in found_features if c in df.columns]

if not numeric_cols:
    print('No feature columns available. Check Cell 3.')
else:
    n_cols = len(numeric_cols)
    ncols_plot = min(3, n_cols)
    nrows_plot = (n_cols + ncols_plot - 1) // ncols_plot
    fig, axes = plt.subplots(nrows_plot, ncols_plot, figsize=(15, 4 * nrows_plot))
    axes = np.array(axes).flatten()
    
    for idx, col in enumerate(numeric_cols):
        ax = axes[idx]
        if actual_label_col:
            for cls in df[actual_label_col].unique():
                subset = df[df[actual_label_col] == cls][col].dropna()
                if len(subset) > 1:
                    subset.plot.kde(ax=ax, label=str(cls), alpha=0.7)
            ax.legend(fontsize=7)
        else:
            df[col].hist(ax=ax, bins=30, color='steelblue', alpha=0.7)
        ax.set_title(col, fontweight='bold')
        ax.set_xlabel(col)
        ax.set_ylabel('Density')
    
    for idx in range(len(numeric_cols), len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('Feature Distributions by Disease Class (KDE)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('feature_distributions.png', dpi=120, bbox_inches='tight')
    plt.show()
    print('Chart saved as feature_distributions.png')

## Cell 8: Correlation Heatmap

NOTE: Correlation measures LINEAR relationships only. A low correlation does not mean the feature is useless — tree-based models (Random Forest, XGBoost) can use non-linear patterns too.


In [ ]:
# CELL 8 — Correlation heatmap

df_corr = df[numeric_cols].copy()
if actual_label_col:
    df_corr['label_encoded'] = pd.Categorical(df[actual_label_col]).codes

corr_matrix = df_corr.corr()
fig, ax = plt.subplots(figsize=(8, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120)
plt.show()

if actual_label_col and 'label_encoded' in corr_matrix.columns:
    label_corr = corr_matrix['label_encoded'].drop('label_encoded').abs().sort_values(ascending=False)
    print('Feature correlation with label (absolute value):')
    for feat, val in label_corr.items():
        bar = '#' * int(val * 30)
        print(f'  {feat:<20} {val:.3f}  {bar}')

## Cell 9: Per-Class Box Plots

Compares median, spread, and outliers of each feature across the five disease classes.


In [ ]:
# CELL 9 — Box plots per class

if not actual_label_col:
    print('Label column not found.')
else:
    n_cols = len(numeric_cols)
    ncols_plot = min(3, n_cols)
    nrows_plot = (n_cols + ncols_plot - 1) // ncols_plot
    fig, axes = plt.subplots(nrows_plot, ncols_plot, figsize=(16, 4 * nrows_plot))
    axes = np.array(axes).flatten()
    palette = sns.color_palette('Set2', df[actual_label_col].nunique())
    
    for idx, col in enumerate(numeric_cols):
        sns.boxplot(data=df, x=actual_label_col, y=col, palette=palette, ax=axes[idx])
        axes[idx].set_title(f'{col}', fontweight='bold')
        axes[idx].set_xlabel('')
        axes[idx].tick_params(axis='x', rotation=30)
    
    for idx in range(len(numeric_cols), len(axes)):
        axes[idx].set_visible(False)
    
    plt.suptitle('Per-Class Feature Box Plots', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('class_boxplots.png', dpi=120, bbox_inches='tight')
    plt.show()

## Cell 10: Outlier Detection (Z-Score)

Values with |Z-score| > 3 are more than 3 standard deviations from the mean. These are flagged — NOT automatically removed.


In [ ]:
# CELL 10 — Outlier detection

print('=== Outlier Detection (Z-score > 3) ===')
for col in numeric_cols:
    z = np.abs(stats.zscore(df[col].dropna()))
    n_out = (z > 3).sum()
    pct   = n_out / len(df) * 100
    flag  = '  <- REVIEW' if pct > 2 else ''
    print(f'  {col:<20}: {n_out:>4} outliers ({pct:.2f}%){flag}')

print()
print('Outliers are NOT removed here. Removal decisions happen in Phase C.')

## Cell 11: Data Leakage Check

Looks for columns that might accidentally encode the class label (ID columns, suspiciously high correlations, temporal columns).


In [ ]:
# CELL 11 — Data leakage check

print('=== Data Leakage Checks ===')

# Check 1: ID-like columns
id_keywords = ['id', 'index', 'sample', 'row', 'image', 'file', 'path', 'name']
id_cols = [c for c in df.columns if any(k in c.lower() for k in id_keywords)]
print(f'ID-like columns: {id_cols if id_cols else "None found"}')
if id_cols:
    print('  -> Verify these do not encode the class. Remove before training if they are identifiers.')

# Check 2: Suspiciously high correlation with label
if actual_label_col:
    lbl = pd.Categorical(df[actual_label_col]).codes
    high_corr_cols = []
    for col in numeric_cols:
        if df[col].std() > 0:
            r = abs(np.corrcoef(df[col].fillna(df[col].mean()), lbl)[0, 1])
            if r > 0.9:
                high_corr_cols.append((col, r))
    if high_corr_cols:
        print(f'HIGH CORRELATION columns (>0.9): {high_corr_cols} <- potential leakage')
    else:
        print('No column has >0.9 correlation with the label. Good.')

# Check 3: Temporal columns
time_keywords = ['date', 'time', 'week', 'month', 'year', 'session', 'season', 'batch']
time_cols = [c for c in df.columns if any(k in c.lower() for k in time_keywords)]
print(f'Temporal columns: {time_cols if time_cols else "None found"}')
if time_cols:
    print('  -> If data is time-ordered, use stratified or time-aware splitting in Phase C.')

## Cell 12: Phase A Summary Report

Auto-generates a summary. Copy this output into your project documentation.


In [ ]:
# CELL 12 — Phase A Summary Report

print('=' * 70)
print('CROPSPECTRA — PHASE A DATASET SUMMARY REPORT')
print('=' * 70)
print(f'Dataset file   : {DATASET_FILE}')
print(f'Total rows     : {len(df)}')
print(f'Total columns  : {len(df.columns)}')
print()
print(f'Features found : {found_features}')
print(f'Features missing: {missing_features}')
print()
if actual_label_col:
    n_cls = df[actual_label_col].nunique()
    print(f'Label column   : {actual_label_col}')
    print(f'Unique classes : {n_cls}')
    print(f'Classes        : {sorted(df[actual_label_col].unique().astype(str))}')
else:
    n_cls = 0
    print('Label column   : NOT FOUND')

total_missing = df.isnull().sum().sum()
n_dup = df.duplicated().sum()
print()
print(f'Missing values : {total_missing}')
print(f'Duplicate rows : {n_dup}')
print()

print('--- FEASIBILITY VERDICT ---')
if actual_label_col and n_cls >= 5 and len(found_features) >= 4:
    print('SUITABLE for Phase B (Exploratory Analysis). Proceed.')
elif actual_label_col and n_cls < 5:
    print(f'WARNING: Only {n_cls} classes found. Expected 5. Check class name spelling.')
elif not actual_label_col:
    print('ERROR: No label column. Cannot train supervised model.')
    print('Update LABEL_COLUMN in Cell 1.')
else:
    print('WARNING: Dataset has issues. Review the outputs above before proceeding.')

print()
print('=' * 70)